In [8]:
!pip install sentence-transformers

In [9]:
!pip install nltk g2p-en numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 4.7 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 50.8 MB/s eta 0:00:0000:01
  Created wheel for distance: filename=Distance-0.1.3-py3-none-any.whl size=16256 sha256=81aeca1e426c2444cd2c2db7b93c3dc0c57d032415a9b79e983127ab541abfa3
  Stored in directory: /root/.cache/pip/wheels/24/a8/58/407063d8e5c1d4dd6594c99d12baa0108570b56a92325587dd
Successfully built distance


In [23]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/kaggle/input/datasets/abhaysharma01702/gpt2-med-large-gensim"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

model.eval()

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50263, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3072, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=1024)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=50263, bias=False)
)

In [28]:
from nltk.corpus import cmudict
cmu_dict = cmudict.dict()

In [29]:
############################################
# SYLLABLE + RHYME UTILITIES
############################################

vowels = "aeiouy"

def fallback_syllables(word):
    word = word.lower()
    count, prev = 0, False
    for c in word:
        if c in vowels:
            if not prev: count += 1
            prev = True
        else:
            prev = False
    if word.endswith("e"): count -= 1
    return max(1, count)

def syllables(word):
    word = word.lower()
    if word in cmu_dict:
        return max(sum(1 for p in pron if str.isdigit(p[-1])) for pron in cmu_dict[word])
    return fallback_syllables(word)

def rhyme_part(word):
    word = word.lower()
    if word not in cmu_dict: return None
    pron = cmu_dict[word][0]
    for i in range(len(pron)-1, -1, -1):
        if pron[i][-1].isdigit(): return tuple(pron[i:])
    return None

def rhyme_score(line, rhyme_word):
    words = line.split()
    if not words: return 0
    r1, r2 = rhyme_part(words[-1].lower()), rhyme_part(rhyme_word)
    return 1 if (r1 and r2 and r1 == r2) else 0

def meter_score(line, target=8):
    return -abs(sum(syllables(w) for w in line.lower().split()) - target)


In [30]:
############################################
# GENERATION
############################################

BAD_WORDS = {"yeah", "oh", "uh", "ayy", "ooh", "mmm"}

def bad_line(line):
    words = line.lower().split()
    if len(words) < 3: return True
    if len(set(words)) / len(words) < 0.4: return True
    if sum(1 for w in words if w in BAD_WORDS) > len(words) / 2: return True
    return False

def sample_line(emotion, max_len=30):
    raw       = model.module if hasattr(model, "module") else model
    prompt    = f"<{emotion}>"
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
    output    = raw.generate(
        input_ids,
        max_new_tokens=max_len,
        do_sample=True,
        top_p=0.9,
        temperature=0.85,
        repetition_penalty=1.25,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text.replace(prompt, "").strip()

def generate_best_line(emotion, rhyme_word=None, meter=8, candidates=5):
    best_line, best_score = None, -999
    for _ in range(candidates):
        line = sample_line(emotion)
        if not line or bad_line(line): continue
        score = meter_score(line, meter) + (rhyme_score(line, rhyme_word) * 5 if rhyme_word else 0)
        if score > best_score:
            best_score, best_line = score, line
    return best_line or sample_line(emotion)

def generate_song(emotion, meter=8):
    raw = model.module if hasattr(model, "module") else model
    raw.eval()
    with torch.no_grad():
        l1 = generate_best_line(emotion, meter=meter)
        l2 = generate_best_line(emotion, meter=meter)
        l3 = generate_best_line(emotion, rhyme_word=l1.split()[-1], meter=meter)
        l4 = generate_best_line(emotion, rhyme_word=l2.split()[-1], meter=meter)
    return "\n".join([l1, l2, l3, l4])


In [31]:
import numpy as np
import re
from nltk.corpus import cmudict
from g2p_en import G2p
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from collections import Counter
from sentence_transformers import SentenceTransformer
import random
random.seed(42)

# =========================
# INIT RESOURCES
# =========================

cmu = cmudict.dict()
g2p = G2p()

VOWEL_PHONEMES = set([
    'AA','AE','AH','AO','AW','AY',
    'EH','ER','EY','IH','IY',
    'OW','OY','UH','UW'
])

# =========================
# UTILITY FUNCTIONS
# =========================

def tokenize(text):
    return re.findall(r"\b\w+'\w+|\w+\b", text.lower())

def get_lines(lyrics):
    return [
        line.strip()
        for line in lyrics.split("\n")
        if line.strip() and not line.strip().startswith("[")
    ]

def is_vowel_phoneme(p):
    return any(p.startswith(v) for v in VOWEL_PHONEMES)

def extract_stress(p):
    if p[-1].isdigit():
        return int(p[-1])
    return 0

# =========================
# PHONEME EXTRACTION
# =========================

def get_phonemes(word):
    word = word.lower()

    # CMU
    if word in cmu:
        return cmu[word][0]

    # G2P
    try:
        phonemes = g2p(word)
        phonemes = [p for p in phonemes if p != ' ']
        if phonemes:
            return phonemes
    except:
        pass

    # FINAL fallback (NEVER return empty)
    return list(word)

# =========================
# PHONEME FEATURES
# =========================

def phoneme_features(p):
    return {
        "is_vowel": is_vowel_phoneme(p),
        "stress": extract_stress(p)
    }

# =========================
# TRANSITION COST (PPFS CORE)
# =========================

def transition_cost(p1, p2, weights):
    f1, f2 = phoneme_features(p1), phoneme_features(p2)

    vowel_switch = int(f1["is_vowel"] != f2["is_vowel"])
    stress_diff = abs(f1["stress"] - f2["stress"])

    return (
        weights["vowel"] * vowel_switch +
        weights["stress"] * stress_diff
    )

# =========================
# 1. PPFS (PHONETIC FLOW)
# =========================

class phonetic_pattern_flow_score:
    def __init__(self, weights=None):
        self.weights = weights or {
            "vowel": 0.6,
            "stress": 0.4
        }

    def compute(self, lyrics):
        words = tokenize(lyrics)

        phonemes = []
        for w in words:
            phonemes.extend(get_phonemes(w))

        if len(phonemes) < 2:
            return 1.0

        costs = [
            transition_cost(phonemes[i], phonemes[i+1], self.weights)
            for i in range(len(phonemes) - 1)
        ]

        avg_cost = np.mean(costs)

        return float(np.exp(-avg_cost))


# =========================
# 2. RSFS (RHYTHM)
# =========================

class rhythmic_structure_flow_score:
    def __init__(self, w_length=0.7, w_stress=0.3):
        self.w_length = w_length
        self.w_stress = w_stress

    def get_syllables_and_stress(self, phonemes):
        syllables = []
        for p in phonemes:
            if p[-1].isdigit():  # vowel phoneme
                syllables.append(int(p[-1]))  # store stress
        return syllables

    def compute(self, lyrics):
        lines = get_lines(lyrics)

        if len(lines) < 2:
            return 1.0

        lengths = []
        stress_patterns = []

        for line in lines:
            words = tokenize(line)

            phonemes = []
            for w in words:
                phonemes.extend(get_phonemes(w))

            syllables = self.get_syllables_and_stress(phonemes)

            if len(syllables) == 0:
                continue

            lengths.append(len(syllables))
            stress_patterns.append(syllables)

        if len(lengths) < 2:
            return 0.0

        # ----------------------
        # Length consistency
        # ----------------------
        target = max(np.median(lengths), 1e-6)
        deviations = [abs(l - target) / target for l in lengths]
        length_score = 1 - np.mean(deviations)

        # ----------------------
        # Stress consistency
        # ----------------------
        stress_diffs = []

        for i in range(len(stress_patterns) - 1):
            p1, p2 = stress_patterns[i], stress_patterns[i+1]

            min_len = min(len(p1), len(p2))

            diff = sum(abs(p1[j] - p2[j]) for j in range(min_len)) / min_len
            stress_diffs.append(diff)

        stress_score = 1 - np.mean(stress_diffs) if stress_diffs else 1.0

        return (
            self.w_length * length_score +
            self.w_stress * stress_score
        )


# =========================
# GLOBAL CONFIG
# =========================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "SamLowe/roberta-base-go_emotions"

FER_LABELS = ["Angry", "Fear", "Happy", "Sad", "Surprise", "Neutral"]

# =========================
# TEXT PROCESSING
# =========================

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\[.*?\]", "", text)  # remove [Chorus], etc.
    text = re.sub(r"[^\w\s']", " ", text)  # remove symbols but keep '
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_lines(lyrics):
    return [
        clean_text(line)
        for line in lyrics.split("\n")
        if clean_text(line)
    ]

# =========================
# LOAD MODEL
# =========================

emotion_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
emotion_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)
emotion_model.eval()

GOEMOTIONS_LABELS = emotion_model.config.id2label

# =========================
# GOEMOTIONS → FER MAPPING
# =========================

FER_MAPPING = {
    "Angry": ["anger", "annoyance", "disapproval", "disgust"],
    "Fear": ["fear", "nervousness"],
    "Happy": ["admiration", "amusement", "approval", "caring", "desire", "excitement", "gratitude", "joy", "love", "optimism", "pride", "relief"],
    "Sad": ["sadness", "disappointment", "grief", "remorse", "embarrassment"],
    "Surprise": ["surprise", "realization"],
    "Neutral": ["neutral", "curiosity", "confusion"]
}

def map_to_fer(go_probs):
    fer_vectors = []

    for probs in go_probs:
        fer_vec = []

        for fer_label in FER_LABELS:
            indices = [
                i for i, label in GOEMOTIONS_LABELS.items()
                if label in FER_MAPPING[fer_label]
            ]
            score = np.sum([probs[i] for i in indices])
            fer_vec.append(score)

        fer_vectors.append(fer_vec)

    return np.array(fer_vectors)

def normalize_fer(fer_vectors):
    fer_vectors = np.array(fer_vectors)
    sums = fer_vectors.sum(axis=1, keepdims=True) + 1e-8
    return fer_vectors / sums

# =========================
# EMOTION PREDICTION (BATCHED)
# =========================

def predict_emotions_batch(lines, batch_size=8):
    all_probs = []

    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]

        inputs = emotion_tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            outputs = emotion_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)

        all_probs.extend(probs.cpu().numpy())

    return np.array(all_probs)

# =========================
# EAS CLASS
# =========================

class emotion_arc_score:
    def __init__(self, use_neutral_weight=True):
        self.use_neutral_weight = use_neutral_weight

    # Jensen-Shannon Divergence
    def js_divergence(self, p, q):
        p = np.clip(p, 1e-8, 1)
        q = np.clip(q, 1e-8, 1)

        m = 0.5 * (p + q)

        return 0.5 * (
            np.sum(p * np.log(p / m)) +
            np.sum(q * np.log(q / m))
        )

    def compute(self, lyrics):
        lines = tokenize_lines(lyrics)

        if len(lines) < 2:
            return 1.0

        # Step 1: Emotion probabilities
        go_probs = predict_emotions_batch(lines)

        # Step 2: Map to FER
        fer_vectors = map_to_fer(go_probs)

        # Step 3: Normalize
        fer_vectors = normalize_fer(fer_vectors)

        # Step 4: Neutral weighting
        if self.use_neutral_weight:
            neutral_idx = FER_LABELS.index("Neutral")
            weights = 1.0 - fer_vectors[:, neutral_idx]
        else:
            weights = np.ones(len(fer_vectors))

        # Step 5: Compute transitions
        distances = []

        for i in range(len(fer_vectors) - 1):
            p = fer_vectors[i]
            q = fer_vectors[i + 1]

            d = self.js_divergence(p, q)

            w = (weights[i] + weights[i + 1]) / 2
            distances.append(w * d)

        if not distances:
            return 1.0

        avg_dist = np.mean(distances)

        # Step 6: Final score
        return float(np.exp(-avg_dist))


class hook_quality_catchiness_score:
    def __init__(self, lambda1=0.4, lambda2=0.3, lambda3=0.3, 
                 n=3, k=3.0, baseline=0.5, temp=1.0):
        self.l1 = lambda1
        self.l2 = lambda2
        self.l3 = lambda3
        self.n = n
        self.k = k
        self.baseline = baseline
        self.temp = temp

    def extract_ngrams(self, words):
        return [" ".join(words[i:i+self.n]) for i in range(len(words)-self.n+1)]

    def softmax(self, x):
        x = x / self.temp
        e = np.exp(x - np.max(x))
        return e / np.sum(e)

    def compute(self, lyrics, ppfs_score=None, rsfs_score=None):
        words = tokenize(lyrics)

        if len(words) < self.n:
            return self.baseline

        ngrams = self.extract_ngrams(words)
        counts = Counter(ngrams)

        repeated = {k: v for k, v in counts.items() if v > 1}

        # ----------------------
        # No hook case
        # ----------------------
        if not repeated:
            if ppfs_score is not None and rsfs_score is not None:
                return (ppfs_score + rsfs_score) / 2
            return self.baseline

        total_ngrams = len(ngrams)

        # Salience
        max_repeat = max(repeated.values())
        salience = max_repeat / total_ngrams

        # Coverage
        total_repeat = sum(repeated.values())
        coverage = total_repeat / total_ngrams

        # Flow
        if ppfs_score is None or rsfs_score is None:
            flow_quality = 0.5
        else:
            flow_quality = (ppfs_score + rsfs_score) / 2

        # Diversity
        unique_ngrams = len(counts)
        diversity = unique_ngrams / total_ngrams
        diversity_penalty = np.exp(-self.k * (1 - diversity))

        # Base score
        base_score = (
            self.l1 * salience +
            self.l2 * coverage +
            self.l3 * flow_quality
        )

        # ----------------------
        # SOFTMAX COMBINATION (YOUR IDEA)
        # ----------------------
        scores = np.array([base_score, diversity_penalty]) * 5
        weights = self.softmax(scores)

        final_score = weights[0] * base_score + weights[1] * diversity_penalty

        return np.clip(final_score, 0.0, 1.0)


class rhyme_consistency_score:
    def __init__(self):
        pass

    def get_last_word(self, line):
        words = tokenize(line)
        return words[-1] if words else ""

    def get_rhyme_part(self, phonemes):
        # Find last stressed vowel
        for i in range(len(phonemes)-1, -1, -1):
            if phonemes[i][-1].isdigit():  # vowel
                return phonemes[i:]
        return phonemes  # fallback

    def rhyme_similarity(self, p1, p2):
        # Compare from end backwards
        i, j = len(p1)-1, len(p2)-1
        match = 0

        while i >= 0 and j >= 0:
            if p1[i] == p2[j]:
                match += 1
            else:
                break
            i -= 1
            j -= 1

        return match / max(len(p1), len(p2))

    def compute(self, lyrics):
        lines = get_lines(lyrics)

        if len(lines) < 2:
            return 0.0

        rhyme_parts = []

        for line in lines:
            word = self.get_last_word(line)

            if not word:
                continue

            phonemes = get_phonemes(word)

            if not phonemes:
                continue

            rhyme = self.get_rhyme_part(phonemes)
            rhyme_parts.append(rhyme)

        if len(rhyme_parts) < 2:
            return 0.0

        scores = []

        for i in range(len(rhyme_parts) - 1):
            sim = self.rhyme_similarity(
                rhyme_parts[i],
                rhyme_parts[i+1]
            )
            scores.append(sim)

        return float(np.mean(scores))


mcs_model = SentenceTransformer('all-MiniLM-L6-v2')

class motif_consistency_score:
    def __init__(self, alpha=0.7, global_sample_ratio=0.3):
        self.alpha = alpha
        self.global_sample_ratio = global_sample_ratio

    def cosine(self, a, b):
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

    def compute(self, lyrics):
        lines = get_lines(lyrics)

        if len(lines) < 2:
            return 1.0

        embeddings = mcs_model.encode(lines)

        # Local similarity
        local_sims = [
            self.cosine(embeddings[i], embeddings[i+1])
            for i in range(len(embeddings)-1)
        ]
        local_score = np.mean(local_sims)

        # Global sampled
        all_pairs = [
            (i, j)
            for i in range(len(embeddings))
            for j in range(i+2, len(embeddings))
        ]

        sample_size = int(len(all_pairs) * self.global_sample_ratio)
        sampled_pairs = random.sample(all_pairs, min(sample_size, len(all_pairs)))

        global_sims = [
            self.cosine(embeddings[i], embeddings[j])
            for i, j in sampled_pairs
        ] if sampled_pairs else [local_score]

        global_score = np.mean(global_sims)

        return self.alpha * local_score + (1 - self.alpha) * global_score


class degeneracy_penality_score:
    def __init__(self, strength=3.0):
        self.strength = strength

    def compute(self, lyrics):
        words = tokenize(lyrics)

        if not words:
            return 1.0

        counts = Counter(words)
        max_freq = max(counts.values())

        ratio = max_freq / len(words)

        return float(np.exp(-self.strength * ratio))

class PHREM:
    def __init__(self, alpha_ppfs = 0.12, beta_rsfs = 0.13, gamma_eas = 0.22, delta_hqcs = 0.13, eta_rcs = 0.1, theta_mcs = 0.2, pi_dps= 0.1):
        self.alpha = alpha_ppfs
        self.beta = beta_rsfs
        self.gamma = gamma_eas
        self.delta = delta_hqcs
        self.eta = eta_rcs
        self.theta = theta_mcs
        self.pi = pi_dps

        self.ppfs = phonetic_pattern_flow_score()
        self.rsfs = rhythmic_structure_flow_score()
        self.eas = emotion_arc_score()
        self.hook = hook_quality_catchiness_score()
        self.rhyme = rhyme_consistency_score()
        self.mcs = motif_consistency_score()
        self.dp = degeneracy_penality_score()

    def compute(self, lyrics):
        ppfs = self.ppfs.compute(lyrics)
        rsfs = self.rsfs.compute(lyrics)
        eas = self.eas.compute(lyrics)
        hook = self.hook.compute(lyrics, ppfs, rsfs)
        rhyme = self.rhyme.compute(lyrics)
        mcs = self.mcs.compute(lyrics)
        dp = self.dp.compute(lyrics)

        final_score = (
            self.alpha * ppfs +
            self.beta * rsfs +
            self.gamma * eas +
            self.delta * hook +
            self.eta * rhyme +
            self.theta * mcs + 
            self.pi*dp
        )

        return final_score

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: SamLowe/roberta-base-go_emotions
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
############################################
# TEST INFERENCE
############################################

for emotion in ["happy", "sad", "angry", "surprise", "fear", "neutral"]:
    print(f"\n{'='*40}")
    print(f"EMOTION: {emotion.upper()}")
    print("="*40)
    print(generate_song(emotion))


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



EMOTION: HAPPY
you are not my friend
i just want to be your guy, i don't need to know what you're doin'
i know you
it's all in my head
i can't believe i'm feeling this way
and you're not here with me, no more
so you've been
trying to
find out
what i'm up against
and i'm trying my best, babe! hey! yeah
i’m a dreamer
i dream of you
i’ve been dreaming since the day that we met, oh
but you

EMOTION: SAD
we’re falling, fallin'
falling from grace
we’ve fallen out of love
fallen from the skies
i think it's time to wake up
to the fact that we've all been sleeping too long in this room, yeah..yeah~yye
all these tears
you know i'm cryin'
all of these memories
they fill my soul with pain and sorrow, yeah
i'm
i've been told that you've been gone for far too long
i've been feeling kinda lonely, but i'm feeling fine
and you've

EMOTION: ANGRY
yuh, yeah
yuh! ayyeeh! what?
a-d-e-l-o! uh-uh!
it’s lil skrr on the beat
yeah i’m still going through the same shit as you, yeah. this is for
no way, oh
no-on

In [33]:
import pandas as pd

phrem = PHREM()

emotions = ["happy", "sad", "angry", "surprise", "fear", "neutral"]

rows = []

num_samples = 100  # total rows you want

for i in range(num_samples):
    emotion = emotions[i % len(emotions)]  # cycle through emotions

    lyrics = generate_song(emotion)
    score = phrem.compute(lyrics)

    rows.append({
        "emotion": emotion,
        "lyrics": lyrics,
        "phrem_score": score
    })

# Create DataFrame
df = pd.DataFrame(rows)

# Save to Excel
df.to_excel("/kaggle/working/generated_lyrics.xlsx", index=False)

print("Saved to generated_lyrics.xlsx")

KeyboardInterrupt: 